# Phutball GRPO pilot: Qwen3.5-9B (thinking on) + LoRA, rewarded only by the engine

The thinking is free-form: the reward checks the final answer with the phutball engine (1 if verified), gives a small
bonus to correct samples shorter than their siblings, and -0.5 to samples cut off at the token cap. No format, language or
readability term, so the thinking may drift ("neuralese"); the drift is logged.

**Runtime:** A100 GPU, High-RAM. **Cost:** smoke test ~0.3 h, pilot 2.5 h (stops itself), eval ~0.7 h: about 3.5 h, ~24 units.
Everything is saved to Drive (`MyDrive/phutball/grpo_pilot/`), and the pilot resumes from its last checkpoint if the
runtime drops: just run the cells again. **Disconnect the runtime when done.**

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os
OUT = '/content/drive/MyDrive/phutball/grpo_pilot'; os.makedirs(OUT, exist_ok=True)
BENCH_OUT = '/content/drive/MyDrive/phutball/llm_bench'          # the zero-shot results (reference)
RUN = f'{OUT}/curriculum2'; CAP = 6144
TOTAL_HOURS = 6.0          # training time across all stages (~6.8 units/h)
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
%cd /content
!test -d phutball-jax || git clone -b expert-dagger https://github.com/echoname6/phutball-jax.git
%cd /content/phutball-jax
!git pull -q && git log --oneline -1
!pip -q install -U "trl[vllm]" peft datasets 2>&1 | tail -3
!pip -q install flash-linear-attention   # fast Gated DeltaNet kernels for Qwen3.5 (else a slow PyTorch fallback)
!pip -q uninstall -y torchao   # Colab ships torchao 0.10; peft refuses versions < 0.16 (and we do not use it)
!python -c "import trl, vllm, peft, transformers, torch; print('trl', trl.__version__, 'vllm', vllm.__version__, 'peft', peft.__version__, 'transformers', transformers.__version__, 'torch', torch.__version__)"

## 1. Smoke test (2 tiny steps, ~15 min incl. the model download)
Catches setup errors before the long run. If it fails, paste the last 40 lines to Claude.

In [ ]:
!python -m llm_train.grpo_pilot --out /content/smoke --smoke 2>&1 | tail -40

## 1b. Self-stop warm start (~45 min, ~5 units)
The model knows short answers early but never stops thinking (forced answers 31/32 right at 12k tokens, 100% of
rollouts cut off). This samples one thinking rollout per puzzle, cuts it at several lengths, forces an answer after
each cut, keeps the shortest cut whose answer the engine verifies and that stays verified at every longer cut, and
fine-tunes briefly on *the model's own thinking up to there + `</think>` + its own answer*. No reasoning is written
for the model; the engine only picks where it could have stopped. The curriculum then starts from this adapter.
Skips whatever is already in Drive.

In [ ]:
WARM = f'{OUT}/self_stop'; os.makedirs(WARM, exist_ok=True)
if not os.path.exists(f'{WARM}/sft.jsonl'):
    !python -m llm_train.self_stop --out {WARM}/sft.jsonl --n 900 2>&1 | grep -v -E "it/s\]|Processed prompts|Adding requests" | tail -25
if not os.path.exists(f'{WARM}/adapter/final/adapter_model.safetensors'):
    !python -m llm_train.sft_stop --data {WARM}/sft.jsonl --out {WARM}/adapter 2>&1 | grep --line-buffered -E "examples|'loss'|saved|rror|Traceback|skipped"
print('warm-start adapter:', os.path.exists(f'{WARM}/adapter/final/adapter_model.safetensors'))

## 2. Puzzle curriculum (stops itself after TOTAL_HOURS of training)
Stages, each starting from the previous stage's LoRA (stage 1 from the self-stop warm start); a stage ends when its finished-and-correct share over the last
5 batches reaches the target, or when its share of the time runs out:
`wins_short` (1-2 jump wins + no-win) >= 50% -> `wins` (all wins) >= 50% -> `wins_place` (+ block, forced) >= 40% -> `full` (+ prevent).
Cut-off rollouts are budget-forced at 25/50/100% of the cap (earlier-correct earns more). Reward = engine-verified
answers only. Safe to re-run after a disconnect: finished stages are skipped and a running stage resumes from its
last checkpoint.

In [ ]:
import json, glob
STAGES = [('wins_short', 0.5), ('wins', 0.5), ('wins_place', 0.4), ('full', 0.0)]
os.makedirs(RUN, exist_ok=True)
def used_hours():
    return sum(json.load(open(f))['elapsed_s'] for f in glob.glob(f'{RUN}/*/stage_done.json')) / 3600
prev = f'{WARM}/adapter/final' if os.path.exists(f'{WARM}/adapter/final/adapter_model.safetensors') else ''
print('starting from:', prev or 'the base model')
for k, (stage, target) in enumerate(STAGES):
    sdir = f'{RUN}/{k + 1}_{stage}'
    if os.path.exists(f'{sdir}/stage_done.json') and os.path.exists(f'{sdir}/final/adapter_model.safetensors'):
        print('done:', stage, open(f'{sdir}/stage_done.json').read()[:120]); prev = f'{sdir}/final'; continue
    left = TOTAL_HOURS - used_hours()
    if left < 0.25: print('time budget used up before', stage); break
    hours = left if stage == 'full' else min(left, max(0.75, left / (len(STAGES) - k)) * 1.5)
    print(f'=== stage {k + 1} {stage}: target {target:.0%}, up to {hours:.2f} h ===')
    init = f'--init-adapter {prev}' if prev else ''
    os.makedirs(sdir, exist_ok=True)
    !python -m llm_train.grpo_pilot --out {sdir} --stage {stage} {init} --advance-correct {target} --hours {hours} --cap {CAP} --force-budgets 0.25,0.5,1 2>&1 | tee -a {sdir}/train.log | grep --line-buffered -E "reward call|\[step|rror|Traceback|saved|skipped|puzzles|forcing|loaded adapter|OutOfMemory"
    if not os.path.exists(f'{sdir}/final/adapter_model.safetensors'): print('stage did not finish: see', f'{sdir}/train.log'); break
    if not os.path.exists(f'{sdir}/stage_done.json'): json.dump({'reason': 'ended', 'elapsed_s': int(hours * 3600)}, open(f'{sdir}/stage_done.json', 'w'))
    prev = f'{sdir}/final'
if prev:
    !rm -rf {RUN}/final && cp -r {prev} {RUN}/final && echo "latest adapter -> {RUN}/final"
print(f'training hours used: {used_hours():.2f}')

## 3. Training curves and the drift of the thinking

In [ ]:
from IPython.display import Image, display
for sdir in sorted(glob.glob(f'{RUN}/[0-9]_*')):
    if not os.path.exists(f'{sdir}/rollouts.jsonl'): continue
    print('#' * 30, sdir.split('/')[-1], '#' * 30)
    !python -m llm_train.analyze_rollouts {sdir}/rollouts.jsonl --plot {sdir}/curves.png --window 3 --puzzles llm_train/puzzles/grpo_train.jsonl
    display(Image(f'{sdir}/curves.png'))

## 4. Eval: untrained vs warm start vs curriculum, same thinking budget (450 benchmark items: 50 per group)
All three are thinking-on at the training cap, budget-forced where cut off, with 95% intervals. The untrained
reference is the zero-shot 24k run cut at the cap (no new generation). Warm start vs curriculum separates what the
self-stop fine-tune did from what GRPO added. ~1 h. Results are copied to RUN/eval/.

In [ ]:
!git pull -q
import subprocess, time, urllib.request
EV = f'{RUN}/eval'; os.makedirs(EV, exist_ok=True); PG = 50
def serve(model, max_len=CAP + 2048, extra=()):
    !pkill -f "vllm serve" ; sleep 5
    log = open('/content/vllm_serve.log', 'w')
    p = subprocess.Popen(['vllm', 'serve', model, '--max-model-len', str(max_len), '--gpu-memory-utilization', '0.90',
                          '--port', '8000', *extra], stdout=log, stderr=subprocess.STDOUT)
    for _ in range(180):
        if p.poll() is not None: break                      # the server exited: do not wait 15 minutes
        try:
            urllib.request.urlopen('http://localhost:8000/health', timeout=5); print('server up:', model, *extra); return p
        except Exception: time.sleep(5)
    !tail -15 /content/vllm_serve.log
    raise RuntimeError('vLLM did not start: see /content/vllm_serve.log')

MODELS = [('warmstart', f'{OUT}/self_stop/adapter/final'), ('curriculum', f'{RUN}/final')]
for name, adapter in MODELS:
    if os.path.exists(f'{EV}/{name}-forced.summary.json'): print('done:', name); continue
    !rm -rf /content/merged && python -m llm_train.merge_lora --adapter {adapter} --out /content/merged 2>&1 | tail -2
    try:
        server = serve('/content/merged'); M = '/content/merged'
    except RuntimeError:                                    # fallback: base model + the LoRA adapter in vLLM
        print('merged model did not serve; trying base + LoRA')
        server = serve('Qwen/Qwen3.5-9B', extra=('--enable-lora', '--max-lora-rank', '32', '--lora-modules', f'{name}={adapter}'))
        M = name
    !python -m llm_bench.run --model {M} --name {name} --thinking true --max-tokens {CAP} --per-group {PG} --concurrency 64 | tail -12
    !python -m llm_bench.run --model {M} --thinking true --salvage {name} --tokenizer Qwen/Qwen3.5-9B --per-group {PG} --concurrency 32 | tail -12
    !cp llm_bench/results/{name}* {EV}/
if not os.path.exists(f'{EV}/qwen3.5-9b-think-at{CAP}-forced.summary.json'):
    server = serve('Qwen/Qwen3.5-9B')
    !cp {BENCH_OUT}/qwen3.5-9b-think.jsonl llm_bench/results/
    !python -m llm_bench.run --model Qwen/Qwen3.5-9B --thinking true --salvage qwen3.5-9b-think --truncate-at {CAP} --per-group {PG} --concurrency 32 | tail -12
    !cp llm_bench/results/qwen3.5-9b-think-at{CAP}* {EV}/
!pkill -f "vllm serve"


In [ ]:
import json
cols = [('untrained', f'{EV}/qwen3.5-9b-think-at{CAP}-forced.summary.json'),
        ('warm start', f'{EV}/warmstart-forced.summary.json'), ('curriculum', f'{EV}/curriculum-forced.summary.json')]
S = {n: json.load(open(p)) for n, p in cols if os.path.exists(p)}
print(f"thinking on, {CAP}-token budget, budget-forced; accuracy [95% interval] (share cut off)")
print(f"{'':26s}" + ''.join(f"{n:>26s}" for n in S))
for k in ('win (all)', 'win: near-miss negative', 'win: positive, 1-jump', 'win: positive, 2-jump', 'win: positive, 3-jump',
          'win: positive, 4-jump', 'forced placement', 'block placement', 'prevent placement'):
    cells = []
    for n in S:
        g = S[n].get(k)
        cells.append(f"{g['accuracy']:6.1%} [{g['ci95'][0]:.0%}-{g['ci95'][1]:.0%}] ({g.get('truncated', 0):.0%})" if g and 'ci95' in g
                     else (f"{g['accuracy']:6.1%}" if g else '-'))
    print(f"{k:26s}" + ''.join(f"{c:>26s}" for c in cells))
print('mean completion tokens   ' + ''.join(f"{S[n].get('mean_completion_tokens', 0):>26.0f}" for n in S))